# 40. Moving-source analysis: amplitude at every station for every second of flight

Peak pressure at a station mixes three things: how strong the rocket was, how far away it was **when
that sound was emitted**, and the direction of the station relative to the exhaust. This notebook
separates them with the forward model of notebook 35:

1. **Trajectory and geometry.** The 2-D gravity-turn trajectory (downrange distance, altitude) is placed
   on the map along `cfg.LAUNCH_AZIMUTH_DEG`. For each station and each source time *t_s* the model gives
   the source–receiver distance *r(t_s)*, the arrival time *t_obs = t_s + r/c*, the emission angle φ
   (between the exhaust direction and the ray to the station) and the ray's elevation angle.
2. **Check the arrival times.** Observed onsets and pressure envelopes are compared with the predicted
   *t_obs(t_s)* curves (record section). Onsets also show which source each station heard first.
3. **Retarded-time amplitudes.** Each station's envelope (RMS in 1-s windows, 0.5–40 Hz) is read at
   *t_obs(t_s)*, for t_s every 0.5 s, giving an amplitude history in **source time**. Multiplying by
   *r(t_s)* removes spherical spreading ("reduced pressure", Pa·km). Gem overflow intervals are masked,
   so near stations contribute only for source times when they were not saturated, and the overlap with
   farther stations ties the network together without a separate peak ladder.
4. **Directivity.** Reduced pressure against emission angle φ, for all stations and source times,
   is an empirical radiation pattern to compare with the model's Gaussian cone.

Assumptions to keep in mind: straight rays at constant sound speed (no wind or refraction), a point
source at the vehicle, the nominal trajectory, and the launch azimuth. Straight rays are reasonable for
the first ~60 s (source below ~10 km) and increasingly doubtful afterwards.

Outputs (`cfg.OUT_DIR`): `moving_source_onsets.csv`, `moving_source_amplitudes.csv`,
`moving_source_record_section.png`, `moving_source_reduced_pressure.png`, `moving_source_directivity.png`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import artemis2_config as cfg
if str(cfg.ROCKET_MODEL_DIR) not in sys.path:
    sys.path.append(str(cfg.ROCKET_MODEL_DIR))
from rocket_infrasound_model.trajectory import GravityTurnParameters, gravity_turn_trajectory

L = cfg.LAUNCHTIME
OUT = cfg.OUT_DIR / "40_moving_source_amplitudes"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'axes.grid': True, 'grid.color': '#dddddd', 'grid.linewidth': 0.6,
                     'axes.spines.top': False, 'axes.spines.right': False})

C_SOUND = 343.0                 # m/s, straight rays
AZ = cfg.LAUNCH_AZIMUTH_DEG     # launch azimuth, deg from north
TS = np.arange(0.0, 150.0 + 1e-9, 0.5)
BAND = (0.5, 40.0)              # Hz
ENV_WIN = 1.0                   # s, RMS envelope window
stations = cfg.station_table()

# One pressure channel per site: unclipped infraBSU where it works, otherwise a Gem.
CHANNELS = ['1R.B14..HDF', '1R.B01.00.DD0', '1R.B03..HDI', '1R.B05.00.DD0', '1R.B12..HDF', '1R.B07..HDF',
            '1R.B23..HDF', '1R.B24..HDF', '1R.B24LT..HDF', '1R.B24RE..HDF', '1R.B29.00.DD0',
            '1R.B20..HDF', '1R.B20W..HDF', '1R.B20S..HDF']

## 1. Trajectory and geometry

In [ ]:
traj_t = np.arange(0.0, TS[-1] + 0.1, 0.1)
s = gravity_turn_trajectory(traj_t, GravityTurnParameters())
traj = pd.DataFrame({k: np.interp(TS, traj_t, s[k]) for k in ('x_m', 'z_m', 'vx_mps', 'vz_mps', 'speed_mps', 'mach')})
traj['t_s'] = TS
traj['east_m'] = traj.x_m * np.sin(np.radians(AZ))
traj['north_m'] = traj.x_m * np.cos(np.radians(AZ))

def geometry(sta, c=C_SOUND):
    """Per source time: distance, arrival time, emission angle and ray elevation for one station."""
    r = stations.loc[sta]
    dx, dy, dz = r.east_m - traj.east_m, r.north_m - traj.north_m, -traj.z_m
    R = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)
    # exhaust direction = minus velocity (3-D, along the launch azimuth)
    sp = np.maximum(traj.speed_mps, 1e-6)
    vx_e, vx_n, vz = traj.vx_mps * np.sin(np.radians(AZ)), traj.vx_mps * np.cos(np.radians(AZ)), traj.vz_mps
    ux, uy, uz = -vx_e / sp, -vx_n / sp, -vz / sp
    ux, uy, uz = (np.where(traj.speed_mps < 1, 0, ux), np.where(traj.speed_mps < 1, 0, uy),
                  np.where(traj.speed_mps < 1, -1, uz))       # on the pad: exhaust straight down
    cosphi = (ux * dx + uy * dy + uz * dz) / R
    return pd.DataFrame(dict(station=sta, t_s=TS, r_m=R, t_obs=TS + R / c,
                             phi_deg=np.degrees(np.arccos(np.clip(cosphi, -1, 1))),
                             elevation_deg=np.degrees(np.arcsin(traj.z_m / R)),
                             source_alt_km=traj.z_m / 1000, source_downrange_km=traj.x_m / 1000))

print(traj.iloc[::20][['t_s', 'x_m', 'z_m', 'speed_mps', 'mach']].round(1).to_string(index=False))

## 2. Data, envelopes, masks

In [ ]:
raw = cfg.launch_stream_raw()
clips = cfg.stream_clip_intervals(raw)
st = cfg.instrument_correct(raw.select(channel='D[D]?') + raw.select(channel='HD?'))
env = {}
for sid in CHANNELS:
    sel = st.select(id=sid)
    if not sel:
        print('missing', sid); continue
    tr = sel[0].copy()
    if tr.stats.sampling_rate > 100:
        tr.filter('lowpass', freq=40, corners=8, zerophase=True)
        tr.resample(100, no_filter=True)
    tr.filter('bandpass', freqmin=BAND[0], freqmax=BAND[1], corners=4, zerophase=True)
    mask = cfg.clip_mask(tr, clips.get(sid, []))
    n = int(ENV_WIN * tr.stats.sampling_rate)
    x2 = np.where(mask, np.nan, tr.data.astype(float) ** 2)
    rms = np.sqrt(pd.Series(x2).rolling(n, center=True, min_periods=int(0.8 * n)).mean().to_numpy())
    noise = np.nanmedian(rms[(tr.times() + (tr.stats.starttime - L) > -600) & (tr.times() + (tr.stats.starttime - L) < -60)])
    env[sid] = dict(t=tr.times() + (tr.stats.starttime - L), rms=rms, noise=noise, sta=tr.stats.station)
print(len(env), 'channels')

## 3. Onsets and sound speed

In [ ]:
rows = []
for sid, e in env.items():
    m = (e['t'] > -5) & (e['t'] < 60) & (e['rms'] > 10 * e['noise'])
    t_on = e['t'][m][0] if m.any() else np.nan
    rows.append(dict(seed_id=sid, station=e['sta'], distance_km=stations.distance_km[e['sta']], onset_s=t_on))
on = pd.DataFrame(rows).sort_values('distance_km')
# Implied emission time of the first audible sound, for straight rays at C_SOUND from the pad.
on['implied_emission_s'] = on.onset_s - on.distance_km * 1000 / C_SOUND
on.to_csv(OUT / 'moving_source_onsets.csv', index=False)
display(on.round(2))
print('Near stations first hear sound emitted several seconds BEFORE liftoff (core-stage engine start);')
print('distant stations first hear the louder booster ignition at ~0 s. A single straight-line fit of')
print('onset vs distance therefore mixes two sources and is not used. Source time 0 = liftoff (cfg.LAUNCHTIME).')
t0 = 0.0

## 4. Record section with predicted arrival curves

In [ ]:
geo = pd.concat([geometry(e['sta']) for e in env.values()], ignore_index=True)
fig, ax = plt.subplots(figsize=(11, 7))
for sid, e in env.items():
    d = stations.distance_km[e['sta']]
    m = (e['t'] >= -5) & (e['t'] <= 200)
    y = e['rms'][m] / np.nanmax(e['rms'][m])
    ax.plot(e['t'][m], d + 0.45 * y, color='#333333', lw=0.6)
dists = np.linspace(0, 8, 50)
for ts in (0, 10, 20, 30, 45, 60, 90, 120):
    sta_d = []
    for sid, e in env.items():
        g = geo[(geo.station == e['sta']) & (geo.t_s == ts)]
        sta_d.append((stations.distance_km[e['sta']], float(g.t_obs.iloc[0]) + t0))
    sta_d.sort()
    ax.plot([b for a, b in sta_d], [a for a, b in sta_d], color='#eb6834', lw=0.8, marker='.', ms=4)
    ax.annotate(f'{ts:g} s', (sta_d[-1][1], sta_d[-1][0] + 0.3), color='#eb6834', fontsize=8, ha='center')
ax.scatter(on.onset_s, on.distance_km, color='#2a78d6', s=18, zorder=4, label='observed onset')
ax.set(xlabel='Time after launch (s)', ylabel='Distance from LC39B (km)', xlim=(-5, 200),
       title=f'RMS envelopes ({BAND[0]:g}-{BAND[1]:g} Hz, gaps = Gem overflow) and predicted arrivals for '
             f'source times shown (c = {C_SOUND:.0f} m/s, azimuth {AZ:g}°)')
ax.legend(frameon=False, fontsize=8, loc='lower right')
fig.tight_layout()
fig.savefig(OUT / 'moving_source_record_section.png', dpi=200)

## 5. Retarded-time amplitudes

In [ ]:
geo = pd.concat([geometry(e['sta']) for e in env.values()], ignore_index=True)
geo['t_obs'] += t0
amp_rows = []
for sid, e in env.items():
    g = geo[geo.station == e['sta']].copy()
    g['seed_id'] = sid
    g['rms_Pa'] = np.interp(g.t_obs, e['t'], e['rms'], left=np.nan, right=np.nan)
    g['snr'] = g.rms_Pa / e['noise']
    g['reduced_Pa_km'] = g.rms_Pa * g.r_m / 1000
    amp_rows.append(g)
amp = pd.concat(amp_rows, ignore_index=True)
amp.loc[amp.snr < 3, ['rms_Pa', 'reduced_Pa_km']] = np.nan
amp.to_csv(OUT / 'moving_source_amplitudes.csv', index=False)
print(amp.groupby('seed_id').apply(lambda d: pd.Series(dict(
    n_valid=d.reduced_Pa_km.notna().sum(), max_reduced=d.reduced_Pa_km.max(),
    ts_of_max=d.t_s[d.reduced_Pa_km.idxmax()] if d.reduced_Pa_km.notna().any() else np.nan))).round(1))

In [ ]:
east = {s for s in stations.index if 0 <= stations.azimuth_deg[s] < 180}
fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for sid, g in amp.groupby('seed_id'):
    sta = sid.split('.')[1]
    col = '#2a78d6' if sta in east else '#eb6834'
    ax[0].semilogy(g.t_s, g.rms_Pa, color=col, lw=0.8)
    ax[1].semilogy(g.t_s, g.reduced_Pa_km, color=col, lw=0.8)
    last = g.dropna(subset=['reduced_Pa_km'])
    if len(last):
        ax[1].annotate(sta, (last.t_s.iloc[-1], last.reduced_Pa_km.iloc[-1]), fontsize=7, color='#333333',
                       xytext=(2, 0), textcoords='offset points')
ax[0].set_ylabel('RMS pressure at arrival (Pa)')
ax[1].set_ylabel('Reduced pressure r·p (Pa·km)')
ax[1].set_xlabel('Source time after launch (s)')
ax[0].plot([], [], color='#2a78d6', label='east of pad (azimuth 0-180°)')
ax[0].plot([], [], color='#eb6834', label='west of pad (azimuth 180-360°)')
ax[0].legend(frameon=False, fontsize=8)
fig.suptitle('Amplitude history in source time; Gem overflow intervals and SNR < 3 omitted', x=0.01, ha='left')
fig.tight_layout()
fig.savefig(OUT / 'moving_source_reduced_pressure.png', dpi=200)

## 6. Empirical directivity

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
v = amp.dropna(subset=['reduced_Pa_km'])
sc = ax[0].scatter(v.phi_deg, v.reduced_Pa_km, c=v.t_s, s=5, cmap='viridis')
ax[0].set(yscale='log', xlabel='Emission angle φ from exhaust axis (deg)', ylabel='Reduced pressure (Pa·km)',
          title='All stations, coloured by source time')
fig.colorbar(sc, ax=ax[0], label='Source time (s)')
bins = np.arange(0, 181, 10)
for (lo, hi), col in zip([(0, 20), (20, 60), (60, 150)], ['#2a78d6', '#1baf7a', '#eb6834']):
    w = v[(v.t_s >= lo) & (v.t_s < hi)]
    med = w.groupby(pd.cut(w.phi_deg, bins), observed=True).reduced_Pa_km.median()
    ax[1].semilogy([iv.mid for iv in med.index], med.values, marker='o', ms=4, color=col, label=f't_s {lo}-{hi} s')
ax[1].set(xlabel='Emission angle φ (deg)', ylabel='Median reduced pressure (Pa·km)', title='Binned by 10°')
ax[1].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'moving_source_directivity.png', dpi=200)

## Notes

* The launch azimuth (`cfg.LAUNCH_AZIMUTH_DEG`) only matters once the vehicle has moved downrange
  (after ~30 s); before that the source is nearly above the pad.
* The trajectory is the nominal gravity-turn model of notebook 35, not the flown trajectory.
* At a fixed source time, stations at different azimuths see the plume at different emission angles,
  so φ and azimuth are entangled; source time separates the effect of the vehicle rising.

## 7. Amplitude field: peak pressure and PGV against distance

Peak |p| (>0.5 Hz) in the first 180 s for every pressure channel, over unclipped samples only. For a
Gem that overflowed, the unclipped peak is a **lower bound** (open symbols) and is left out of the
power-law fit. Peak vertical ground velocity (>1 Hz) for Trillium `DHZ` and SmartSolo `DPZ` (SmartSolo
sign reversed, notebook 50). B03's Centaur channels (hardware fault) and B05's Trillium are flagged.

In [ ]:
win = (L, L + 180)
st_all = cfg.instrument_correct(raw)
for tr in st_all:
    if tr.stats.location == '--':
        tr.stats.location = ''
over_ids = {sid for sid, ivs in clips.items() if ivs}
prow = []
for tr in st_all:
    cha = tr.stats.channel
    if not (cha.startswith('HD') or cha in cfg.INFRABSU_CHANNELS or cha in ('DHZ', 'DPZ')):
        continue
    sta = tr.stats.station
    if sta not in stations.index:
        continue
    t = tr.copy()
    fmin = 0.5 if (cha.startswith('HD') or cha.startswith('DD')) else 1.0
    t.filter('highpass', freq=fmin, corners=4, zerophase=True)
    m = ~cfg.clip_mask(t, clips.get(tr.id, []))
    seg = (t.times() + (t.stats.starttime - L) >= 0) & (t.times() + (t.stats.starttime - L) <= 180)
    x = t.data[seg & m]
    noise = t.slice(L - 900, L - 720).data
    if not len(x):
        continue
    pk = np.abs(x).max()
    prow.append(dict(seed_id=tr.id, station=sta, channel=cha, distance_km=stations.distance_km[sta],
                     azimuth_deg=stations.azimuth_deg[sta],
                     kind='pressure' if fmin == 0.5 else 'velocity',
                     instrument={'H': 'Gem', 'D': 'infraBSU'}[cha[0]] if fmin == 0.5 else ('Trillium' if cha == 'DHZ' else 'SmartSolo'),
                     peak=pk, snr=pk / max(np.abs(noise).max(), 1e-12),
                     t_peak_s=float(t.times()[seg & m][np.argmax(np.abs(x))] + (t.stats.starttime - L)),
                     lower_bound=tr.id in over_ids))
amp_f = pd.DataFrame(prow)
dead = amp_f.snr < 5          # unconnected Centaur inputs, dead sensors
fault = amp_f.seed_id.str.match(r'1R\.B03\.00\.') | (amp_f.seed_id == '1R.B05.00.DHZ')
amp_f['use_in_fit'] = ~dead & ~fault & ~amp_f.lower_bound
amp_f['flag'] = np.select([dead, fault, amp_f.lower_bound], ['no signal', 'faulty', 'overflow: lower bound'], '')
amp_f = amp_f[~dead].sort_values(['kind', 'distance_km'])
amp_f.to_csv(OUT / 'amplitude_field.csv', index=False)

fits = {}
for kind in ('pressure', 'velocity'):
    g = amp_f[(amp_f.kind == kind) & amp_f.use_in_fit]
    b, a = np.polyfit(np.log10(g.distance_km), np.log10(g.peak), 1)
    res = np.log10(g.peak) - (a + b * np.log10(g.distance_km))
    fits[kind] = (a, b, res.std())
    print(f"{kind}: peak = {10 ** a:.3g} * r_km^{b:.2f}  (n = {len(g)}, scatter {res.std():.2f} log10 units)")
amp_f['residual_log10'] = np.log10(amp_f.peak) - [fits[k][0] + fits[k][1] * np.log10(d)
                                                  for k, d in zip(amp_f.kind, amp_f.distance_km)]
amp_f.to_csv(OUT / 'amplitude_field.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
MK = {'infraBSU': ('o', '#2a78d6'), 'Gem': ('^', '#eb6834'), 'Trillium': ('o', '#2a78d6'), 'SmartSolo': ('s', '#1baf7a')}
for ax, kind, unit in ((axes[0], 'pressure', 'Peak pressure (Pa), >0.5 Hz'), (axes[1], 'velocity', 'PGV (m/s), vertical, >1 Hz')):
    g = amp_f[amp_f.kind == kind]
    for inst, h in g.groupby('instrument'):
        mk, col = MK[inst]
        for lb, hh in h.groupby(h.lower_bound | (h.flag == 'faulty')):
            ax.scatter(hh.distance_km, hh.peak, marker=mk, s=40, zorder=3,
                       facecolors='none' if lb else col, edgecolors=col,
                       label=inst + (' (lower bound / faulty)' if lb else ''))
    for _, r in g.iterrows():
        ax.annotate(r.station, (r.distance_km, r.peak), fontsize=6, xytext=(3, 2), textcoords='offset points')
    a, b, sd = fits[kind]
    rr = np.linspace(0.6, 8.5, 50)
    ax.plot(rr, 10 ** (a + b * np.log10(rr)), color='#333333', lw=1, label=f'fit: ∝ r$^{{{b:.2f}}}$')
    ax.set(xscale='log', yscale='log', xlabel='Distance from LC39B (km)', ylabel=unit)
    ax.legend(frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(OUT / 'amplitude_field.png', dpi=200)
with pd.option_context('display.float_format', '{:.3g}'.format, 'display.width', 200):
    display(amp_f[['seed_id', 'distance_km', 'azimuth_deg', 'peak', 't_peak_s', 'flag', 'residual_log10']])

## 8. Directivity fitted to the data, and the forward model re-run at the real stations

The Gaussian cone *D(φ) = exp[−½((φ − φ₀)/σ)²]* is fitted to the reduced pressures of Section 6 for
source times after the pad phase (15–60 s, where straight rays are reasonable; 15–120 s for comparison).
The forward model (point source on the trajectory, *Q(t) = 1 − e^{−t/8 s}*, straight rays) is then
evaluated at the real station positions with the original cone (φ₀ = 30°, σ = 20°) and the fitted one.
For each station, the synthetic envelope's peak time and amplitude are compared with the observed RMS
envelope's (source times ≥ 10 s, overflow masked). Amplitudes are compared after one common scale
factor, so only the relative pattern is tested. Because the cone is fitted to these same envelopes, the
amplitude comparison is a consistency check; the timing comparison is closer to an independent test.

In [ ]:
from scipy.optimize import curve_fit

def log_cone(phi, c, phi0, sig):
    return c - 0.5 * ((phi - phi0) / sig) ** 2 / np.log(10)

cone_fits = {}
for lo, hi in ((15, 60), (15, 120)):
    v = amp[(amp.t_s >= lo) & (amp.t_s <= hi)].dropna(subset=['reduced_Pa_km'])
    p, cov = curve_fit(log_cone, v.phi_deg, np.log10(v.reduced_Pa_km), p0=(2.0, 60.0, 30.0))
    err = np.sqrt(np.diag(cov))
    res = np.log10(v.reduced_Pa_km) - log_cone(v.phi_deg, *p)
    cone_fits[(lo, hi)] = p
    print(f"t_s {lo}-{hi} s: phi0 = {p[1]:.1f} ± {err[1]:.1f} deg, sigma = {abs(p[2]):.1f} ± {err[2]:.1f} deg, "
          f"n = {len(v)}, scatter {res.std():.2f} log10")
PHI0_FIT, SIG_FIT = cone_fits[(15, 60)][1], abs(cone_fits[(15, 60)][2])

In [ ]:
def synthetic(sta, phi0, sig, tau=8.0):
    g = cfg.source_geometry(sta, ts=np.arange(0.0, 150.0, 0.1), c=C_SOUND)
    q = 1 - np.exp(-g.t_s / tau)
    a = q * np.exp(-0.5 * ((g.phi_deg - phi0) / sig) ** 2) / (g.r_m / 1000)
    keep = g.t_s >= 10
    i = int(np.argmax(np.where(keep, a, -1)))
    return g.t_obs.iloc[i] + t0, a.iloc[i], g.t_s.iloc[i], g.phi_deg.iloc[i]

obs = []
for sid, e in env.items():
    sta = e['sta']
    t_ts10 = cfg.source_geometry(sta, ts=np.array([10.0]), c=C_SOUND).t_obs.iloc[0]
    m = (e['t'] >= t_ts10) & (e['t'] <= 200) & np.isfinite(e['rms'])
    i = np.argmax(np.where(m, e['rms'], -1))
    obs.append(dict(seed_id=sid, station=sta, distance_km=stations.distance_km[sta],
                    azimuth_deg=stations.azimuth_deg[sta], t_obs_peak=e['t'][i], env_peak=e['rms'][i]))
cmp = pd.DataFrame(obs)
summary = []
for name, (phi0, sig) in {'original (30°, 20°)': (30.0, 20.0),
                          f'fitted ({PHI0_FIT:.0f}°, {SIG_FIT:.0f}°)': (PHI0_FIT, SIG_FIT)}.items():
    syn = [synthetic(s, phi0, sig) for s in cmp.station]
    cmp[f't_syn {name}'] = [x[0] for x in syn]
    a_syn = np.array([x[1] for x in syn])
    scale = np.median(np.log10(cmp.env_peak) - np.log10(a_syn))
    cmp[f'a_syn {name}'] = 10 ** (np.log10(a_syn) + scale)
    dt = cmp.t_obs_peak - cmp[f't_syn {name}']
    dlog = np.log10(cmp.env_peak) - np.log10(cmp[f'a_syn {name}'])
    east = cmp.azimuth_deg < 180
    def slope(y):
        return np.polyfit(np.log10(cmp.distance_km), np.log10(y), 1)[0]
    def speed(t):
        return 1000 / np.polyfit(cmp.distance_km, t, 1)[0]
    summary.append(dict(model=name, rms_time_resid_s=np.sqrt(np.mean(dt ** 2)), mean_time_resid_s=dt.mean(),
                        rms_log_amp_resid=np.sqrt(np.mean(dlog ** 2)),
                        west_east_ratio_syn=10 ** (np.log10(cmp[f'a_syn {name}'][~east] * cmp.distance_km[~east]).mean()
                                                   - np.log10(cmp[f'a_syn {name}'][east] * cmp.distance_km[east]).mean()),
                        decay_exponent_syn=slope(cmp[f'a_syn {name}']),
                        apparent_speed_syn=speed(cmp[f't_syn {name}'])))
east = cmp.azimuth_deg < 180
summary.append(dict(model='observed', west_east_ratio_syn=10 ** (np.log10(cmp.env_peak[~east] * cmp.distance_km[~east]).mean()
                                                                  - np.log10(cmp.env_peak[east] * cmp.distance_km[east]).mean()),
                    decay_exponent_syn=np.polyfit(np.log10(cmp.distance_km), np.log10(cmp.env_peak), 1)[0],
                    apparent_speed_syn=1000 / np.polyfit(cmp.distance_km, cmp.t_obs_peak, 1)[0]))
model_cmp = pd.DataFrame(summary).rename(columns={'west_east_ratio_syn': 'west_east_ratio (r-corrected)',
                                                   'decay_exponent_syn': 'decay_exponent',
                                                   'apparent_speed_syn': 'apparent_speed_mps'})
model_cmp.to_csv(OUT / 'model_data_comparison.csv', index=False)
cmp.to_csv(OUT / 'model_data_stations.csv', index=False)
with pd.option_context('display.float_format', '{:.2f}'.format, 'display.width', 250):
    display(model_cmp)
    display(cmp.round(2))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
v = amp[(amp.t_s >= 15) & (amp.t_s <= 120)].dropna(subset=['reduced_Pa_km'])
ax[0].scatter(v.phi_deg, v.reduced_Pa_km, s=4, c=v.t_s, cmap='viridis', alpha=0.6)
pp = np.linspace(0, 95, 200)
c_fit = cone_fits[(15, 60)][0]
ax[0].plot(pp, 10 ** log_cone(pp, c_fit, PHI0_FIT, SIG_FIT), color='#333333', lw=1.5,
           label=f'fit: φ₀ = {PHI0_FIT:.0f}°, σ = {SIG_FIT:.0f}°')
c_orig = np.median(np.log10(v.reduced_Pa_km) - log_cone(v.phi_deg, 0, 30, 20))
ax[0].plot(pp, 10 ** log_cone(pp, c_orig, 30, 20), color='#eb6834', lw=1.2, ls='--', label='original: 30°, 20°')
ax[0].set(yscale='log', xlabel='Emission angle φ (deg)', ylabel='Reduced pressure (Pa·km)',
          title='Directivity, source times 15-120 s')
ax[0].legend(frameon=False, fontsize=8)
names = [c[6:] for c in cmp.columns if c.startswith('t_syn ')]
styles = {names[0]: ('#eb6834', 's'), names[1]: ('#2a78d6', 'o')}
for n in names:
    col, mk = styles[n]
    ax[1].scatter(cmp.t_obs_peak, cmp[f't_syn {n}'], color=col, marker=mk, s=30, label=n)
    ax[2].scatter(cmp.env_peak, cmp[f'a_syn {n}'], color=col, marker=mk, s=30, label=n)
lim = [0, max(cmp.t_obs_peak.max(), max(cmp[f't_syn {n}'].max() for n in names)) * 1.05]
ax[1].plot(lim, lim, color='#999999', lw=0.8)
ax[1].set(xlabel='Observed envelope peak time (s)', ylabel='Synthetic peak time (s)', title='Peak arrival time')
alim = [cmp.env_peak.min() / 1.5, cmp.env_peak.max() * 1.5]
ax[2].plot(alim, alim, color='#999999', lw=0.8)
ax[2].set(xscale='log', yscale='log', xlabel='Observed envelope peak (Pa)', ylabel='Synthetic peak (scaled)',
          title='Peak amplitude')
for a in ax[1:]:
    a.legend(frameon=False, fontsize=8)
for _, r in cmp.iterrows():
    ax[1].annotate(r.station, (r.t_obs_peak, r[f't_syn {names[1]}']), fontsize=6, xytext=(3, 2), textcoords='offset points')
fig.tight_layout()
fig.savefig(OUT / 'model_data_directivity.png', dpi=200)